# 42 — Multimodal RAG

Describe images with a **vision LLM**, store the text descriptions in **ChromaDB**, then answer natural-language questions via **similarity search** over those descriptions.

**What you'll learn**
- How to encode images as base64 and pass them to GPT-4o-mini with `image_url` content blocks
- Why converting images to text descriptions beats raw CLIP embeddings for natural-language queries
- How to build a 3-node LangGraph pipeline: describe → store → answer
- ChromaDB `from_texts` + `similarity_search` for in-memory retrieval
- The `MultimodalState` TypedDict flowing through LangGraph nodes

**Companion examples:** 1-basic-local-rag (Chroma basics), 17-corrective-rag (graded retrieval)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q chromadb langchain-community langchain-openai httpx python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. How Multimodal RAG works ────────────────────────────────────────────────
# Two strategies for querying image collections with natural language:
#
# Strategy A — CLIP (direct image embeddings)
#   image → CLIP encoder → embedding → vector store
#   + No LLM needed at index time
#   - Embeddings are in image-space; natural-language queries work less well
#   - Hard to index fine-grained details (text in images, object relationships)
#
# Strategy B — Description indexing (this example)
#   image → vision LLM → text description → text embedding → vector store
#   + Descriptions are in language-space; semantic search works perfectly
#   + Rich detail: colors, objects, text, spatial relationships
#   - Costs one vision LLM call per image at index time
#
# Graph pipeline:
#   START
#     → describe  : fetch each image, call GPT-4o-mini vision, get description
#     → store     : embed descriptions, load into Chroma in-memory collection
#     → answer    : similarity_search(query, k=3), summarise with QA LLM
#   END

diagram = """
IMAGE_DOCS (list of {id, url, label})
       |
       v
 [describe] -- vision LLM per image --> descriptions: list[str]
       |
       v
  [store]   -- Chroma.from_texts() --> _vectorstore (in-memory)
       |
       v
 [answer]   -- similarity_search(query, k=3) --> answer: str
"""
print(diagram)

In [ ]:
# ── 4. Full self-contained implementation ─────────────────────────────────────
import base64
from typing import TypedDict

import httpx
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, StateGraph

# ── Image catalogue ────────────────────────────────────────────────────────────
IMAGE_DOCS = [
    {
        "id": "cat",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
        "label": "a cat",
    },
    {
        "id": "dog",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
        "label": "a dog",
    },
    {
        "id": "coffee",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/A_small_cup_of_coffee.JPG/320px-A_small_cup_of_coffee.JPG",
        "label": "a cup of coffee",
    },
]

QUERIES = [
    "What animal is in the images?",
    "Is there a beverage shown?",
]

# ── State ──────────────────────────────────────────────────────────────────────
class MultimodalState(TypedDict):
    descriptions: list[str]
    query: str
    answer: str

# ── Clients ────────────────────────────────────────────────────────────────────
vision_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=200)
qa_llm     = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Helper ─────────────────────────────────────────────────────────────────────
def fetch_image_b64(url: str) -> str:
    resp = httpx.get(url, timeout=15, follow_redirects=True)
    return base64.b64encode(resp.content).decode()

# ── Nodes ──────────────────────────────────────────────────────────────────────
_vectorstore: Chroma | None = None

def describe_images(state: MultimodalState) -> dict:
    descriptions = []
    for doc in IMAGE_DOCS:
        try:
            b64 = fetch_image_b64(doc["url"])
            # image_url content block: pass base64-encoded JPEG to GPT-4o-mini vision
            msg = HumanMessage(content=[
                {"type": "text", "text": f"Describe this image in 2 sentences. Label: {doc['label']}"},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ])
            result = vision_llm.invoke([msg])
            desc = f"[{doc['id']}] {result.content}"
        except Exception as e:
            desc = f"[{doc['id']}] {doc['label']} (description unavailable: {e})"
        descriptions.append(desc)
        print(f"  Described: {doc['id']}")
    return {"descriptions": descriptions}

def store_descriptions(state: MultimodalState) -> dict:
    global _vectorstore
    _vectorstore = Chroma.from_texts(
        state["descriptions"],
        embedding=embeddings,
        collection_name="image_descriptions",
    )
    print(f"  Stored {len(state['descriptions'])} descriptions in Chroma")
    return {}

def answer_query(state: MultimodalState) -> dict:
    if _vectorstore is None:
        return {"answer": "No descriptions indexed yet."}
    docs = _vectorstore.similarity_search(state["query"], k=3)
    context = "\n".join(d.page_content for d in docs)
    prompt = f"Context:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    result = qa_llm.invoke([HumanMessage(content=prompt)])
    return {"answer": result.content}

# ── Graph ──────────────────────────────────────────────────────────────────────
def create_workflow():
    graph = StateGraph(MultimodalState)
    graph.add_node("describe", describe_images)
    graph.add_node("store", store_descriptions)
    graph.add_node("answer", answer_query)
    graph.set_entry_point("describe")
    graph.add_edge("describe", "store")
    graph.add_edge("store", "answer")
    graph.add_edge("answer", END)
    return graph.compile()

app = create_workflow()
print("Graph compiled: START → describe → store → answer → END")

In [ ]:
# ── 5. Run queries ─────────────────────────────────────────────────────────────
# Note: describe_images runs once per invoke, re-fetching and re-describing each image.
# In production you would index once, then query many times (separate index/query phases).

for query in QUERIES:
    print(f"\nQUERY: {query}")
    result = app.invoke({"descriptions": [], "query": query, "answer": ""})
    print(f"ANSWER: {result['answer'][:300]}")
    print(f"\nDescriptions indexed:")
    for d in result["descriptions"]:
        print(f"  {d[:120]}")

## Exercises

**Exercise 1 — Add more images:** Extend `IMAGE_DOCS` with 2 new entries (any public image URLs). Re-run and verify the new images appear in descriptions and influence answers.

**Exercise 2 — Store URL in metadata:** Modify `store_descriptions` to pass `metadatas=[{"url": doc["url"]} for doc in IMAGE_DOCS]` to `Chroma.from_texts`. Then in `answer_query`, print `doc.metadata["url"]` alongside each retrieved description so answers include a source link.

**Exercise 3 — Compare with and without the store step:** Comment out the `store → answer` edge and wire `describe → answer` directly. In `answer_query`, fall back to joining `state["descriptions"]` as context instead of using the vectorstore. Compare answer quality — does similarity search add value with only 3 images?

**Exercise 4 — Try DALL-E generated images:** Use the OpenAI Images API to generate an image (`client.images.generate(...)`), pass the returned URL directly into `fetch_image_b64`, and add it to your index dynamically.

## What's next

- **1-basic-local-rag** — ChromaDB and similarity search fundamentals without the vision layer
- **17-corrective-rag** — Grade retrieved documents for relevance before generating an answer